# Meet the wines: curation and exploration

Goal: understand the raw data well enough to know what a good price prediction would even mean.

The dataset is [`spawn99/wine-reviews`](https://huggingface.co/datasets/spawn99/wine-reviews):
Wine Enthusiast tasting notes with a critic score (`points`), a price, and geography. Two Kaggle
scrapes merged together, so expect duplicates and missing prices.

Its train/validation/test split is one arbitrary partition of that single scrape, so all three are
concatenated here and re-split at the end -- otherwise a third of the reviews go unused, and
duplicate notes spanning the upstream splits survive deduplication.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from datasets import concatenate_datasets, load_dataset

from pricer.curate import balance, deduplicate, holdout, price_histogram
from pricer.items import Wine
from pricer.loaders import RAW_DATASET
from pricer.parser import compose, parse

raw = concatenate_datasets(list(load_dataset(RAW_DATASET).values()))
raw

### One row, in full

In [ ]:
for key, value in raw[0].items():
    print(f"{key:>14}: {value}")

### How much of the raw data is usable?

`parse` rejects a row when the price is missing, the price is outside \$4-\$500, or the tasting note
is under 100 characters. Everything else becomes a `Wine`.

In [ ]:
parsed = [parse(row) for row in raw]
wines = [wine for wine in parsed if wine]
print(f"{len(raw):,} raw rows -> {len(wines):,} usable wines ({len(wines) / len(raw):.0%})")

### Duplicates

The merge left the same tasting note in the file many times over. Left alone, the identical wine
appears in both train and test, and every model looks better than it is.

In [ ]:
notes = Counter(wine.description for wine in wines)
print(f"{len(notes):,} distinct notes among {len(wines):,} wines")
for note, count in notes.most_common(3):
    print(f"\n{count} copies: {note[:110]}...")

In [ ]:
wines = deduplicate(wines)

### The target

Price is log-normal. The median bottle is cheap and the tail is long, which is why the metric that
matters here is RMSLE (error in *relative* terms) rather than raw-dollar MSE -- being \$20 out on a
\$25 bottle is a disaster, on a \$400 bottle it is noise.

In [ ]:
prices = np.array([wine.price for wine in wines])
print(f"median ${np.median(prices):,.0f}  mean ${prices.mean():,.0f}  p95 ${np.percentile(prices, 95):,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(prices, bins=80, color="#7f1d3f")
axes[0].set_title("price ($)")
axes[1].hist(np.log1p(prices), bins=80, color="#7f1d3f")
axes[1].set_title("log1p(price)")
plt.show()

### Does the critic score explain the price?

Partly -- and that is exactly why `points` is kept out of the text the models read by default. It is
a strong, easy proxy that would let a model skip reading the tasting note. Turn it back on with
`compose(wine, fields=(..., "points"))` when you want to measure how much of the signal it carries.

In [ ]:
points = np.array([wine.points for wine in wines])
correlation = np.corrcoef(points, np.log1p(prices))[0, 1]
plt.figure(figsize=(7, 4))
plt.scatter(points + np.random.uniform(-0.4, 0.4, len(points)), prices, s=2, alpha=0.05, color="#7f1d3f")
plt.yscale("log")
plt.title(f"points vs price (r = {correlation:.2f} on log price)")
plt.show()

### Where the wines come from, and what they are

In [ ]:
facets = {
    "country": lambda wine: wine.country,
    "variety": lambda wine: wine.variety,
    "taster": lambda wine: wine.taster,
}
for label, pick in facets.items():
    values = [pick(wine) or "unknown" for wine in wines]
    counts = Counter(values)
    print(f"\n{label}: {len(counts):,} distinct")
    for name, count in counts.most_common(8):
        median = np.median([wine.price for wine, value in zip(wines, values, strict=True) if value == name])
        print(f"  {name[:28]:<30}{count:>7,}   median ${median:>6,.0f}")

### Tasting notes are short

A few hundred characters each, so a fine-tune sees the whole note comfortably inside a small context.

In [ ]:
lengths = np.array([len(wine.description) for wine in wines])
print(f"median {np.median(lengths):.0f} chars, p99 {np.percentile(lengths, 99):.0f} chars")
plt.figure(figsize=(7, 3))
plt.hist(lengths, bins=80, color="#7f1d3f")
plt.title("tasting note length (chars)")
plt.show()

### Hold out the test set first, then balance

Trained on the raw distribution a model learns that guessing \$25 is nearly always safe. Capping how
many wines each log-price bin may contribute flattens the target and forces the model to read.

Order matters. Validation and test come out of the deduplicated pool *before* balancing, so the cap
only ever changes the training set -- balance first and every cap would be scored on a differently
shaped test set, mixing "more data" with "easier exam". The unbalanced test set is also the honest
one: wines in a shop are not uniform in price.

The trade is volume: the top bins hold only a few hundred wines each, so a *perfectly* flat set would
be tiny. `cap=10_000` keeps a bit over half the data. **Experiment:** rerun with `cap=20_000` and
compare RMSLE on the expensive half of the test set -- now a fair comparison.

In [ ]:
price_histogram(wines)

In [ ]:
pool, val, test = holdout(wines)
train = balance(pool, cap=10_000)
price_histogram(train)

### What a model actually reads

In [ ]:
print(train[0].full)
print("\n--- price:", train[0].price)

In [ ]:
# The same wine with the leaky fields switched on, for comparison
print(compose(train[0], fields=("vintage", "variety", "country", "region", "winery", "points", "note")))

### Cache the splits

In [ ]:
Wine.save_local(train=train, val=val, test=test)

Next: `scripts/baselines.py` fits the classical ladder on this cache, and
`notebooks/2_baseline_ladder.ipynb` walks through what each rung is worth.